# Distance to TLS and bronchi — visualization
Visualization of the distance-to-TLS / distance-to-bronchi analysis. This notebook loads the output of [`02_distance_tls_bronchi.ipynb`](02_distance_tls_bronchi.ipynb), which provides the signed `distance_to_tls` (negative inside the TLS), `avg_distance_to_bronchi_zone`, and the gated `spatial_region` as pre-computed columns.

**Pinned Environment:** [`conda_envs/space2_20250604.yml`](../conda_envs/space2_20250604.yml)  

In [ ]:
from pathlib import Path
import sys
import os
import time
import warnings
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata
import scanpy as sc
import squidpy as sq


import scipy
from scipy.spatial import distance_matrix
from scipy.spatial.distance import cdist
from scipy import stats
import scipy.ndimage as ndi
from scipy.stats import gaussian_kde
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks

from shapely.geometry import Point

from shapely.geometry import MultiPolygon
from alphashape import alphashape
from alphashape import optimizealpha
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer

import pickle
import joblib

import random
import matplotlib.colors as mcolors

import matplotlib as mpl
mpl.rcParams['axes.titlesize'] = 24
mpl.rcParams['pdf.fonttype'] = 42

import warnings
warnings.simplefilter(action='ignore', category=Warning)

sys.version_info

## Local file info

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[0]))

from config.paths import BASE_OUTDIR, INPUTS_DIR, FUNCTIONS_DIR

inputs_dir = INPUTS_DIR
dist_out_dir = os.path.join(BASE_OUTDIR, "downstream_analysis/distance")


plot_out_dir = os.path.join(dist_out_dir, 'plots')

print(inputs_dir)
print(dist_out_dir)

In [ ]:
import importlib
if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.append(str(FUNCTIONS_DIR))

import plotting_utils as pu
import de_utils as du

import distance_utils
importlib.reload(distance_utils)
from distance_utils import plot_celltype_density_2d, scvelo_heatmap, distance_xy_kde


In [ ]:
importlib.reload(du)

## Load adata
Loads the output of `02_distance_tls_bronchi.ipynb`.

In [ ]:
adata = sc.read_h5ad(os.path.join(dist_out_dir, 'adata_distance_zones_structure_update2026.h5ad'))

## Load palettes

In [ ]:
with open(os.path.join(inputs_dir, '20250519_celltype_palette_mapped.pkl'), 'rb') as f:
    celltype_palette_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, '20250520_celltype_palette_coarse_mapped.pkl'), 'rb') as f:
    celltype_palette_coarse_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, '20250520_celltype_palette_coarser_mapped.pkl'), 'rb') as f:
    celltype_palette_coarser_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, '20250521_zone_consol_palette_mapped.pkl'), 'rb') as f:
    zone_consol_palette_mapped = pickle.load(f)

In [ ]:
import matplotlib.colors as clr
import colorcet

zissou = [
    "#3A9AB2",
    "#6FB2C1",
    "#91BAB6",
    "#A5C2A3",
    "#BDC881",
    "#DCCB4E",
    "#E3B710",
    "#E79805",
    "#EC7A05",
    "#EF5703",
    "#F11B00",
]

colormap = clr.LinearSegmentedColormap.from_list("Zissou", zissou)
colormap_r = clr.LinearSegmentedColormap.from_list("Zissou", zissou[::-1])

## Spatial plots of distance

In [ ]:
sample_label = 'HDM_day3'
adata_d3 = adata[adata.obs['sample_label'] == sample_label, :]
adata_plot = adata_d3.copy()
adata_plot.obs['zone_consol_temp'] = adata_plot.obs['zone_consol'].copy()
adata_plot.obs.loc[adata_plot.obs['zone_consol'] != 'TLS', 'zone_consol_temp'] = np.nan
sc.pl.embedding(adata_plot, basis='spatial', color='zone_consol_temp',
                palette = zone_consol_palette_mapped,
                frameon=False, 
                size=50, 
                title = '',
                legend_loc = 'none',
                show=False)

ax = plt.gca()
tls_poly = joblib.load(os.path.join(dist_out_dir, 'tls_poly_' + sample_label + '.pkl'))
bronchi_poly = joblib.load(os.path.join(dist_out_dir, 'bronchi_poly_' + sample_label + '.pkl'))
vessels_poly = joblib.load(os.path.join(dist_out_dir, 'vessels_poly_' + sample_label + '.pkl'))
for p in tls_poly.geoms:
        ax.plot(*p.exterior.xy, c='black', linewidth=4)
   
plt.xlim([3700, 4400])
plt.ylim([4200, 4900])
# plt.savefig(os.path.join(plot_out_dir, "zone_day3_tls_zoom.png"), dpi=300, bbox_inches='tight', transparent=True)

fig = sc.pl.embedding(adata_d3, basis='spatial', 
                # color='avg_distance_to_TLS_zone',
                color = 'distance_to_tls',
                cmap = colormap_r,
                vmax=450,
                frameon=False, 
                size=50, 
                title = '',
                # colorbar_loc = None,
                show=False, return_fig=True)

ax = plt.gca()
tls_poly = joblib.load(os.path.join(dist_out_dir, 'tls_poly_' + sample_label + '.pkl'))
bronchi_poly = joblib.load(os.path.join(dist_out_dir, 'bronchi_poly_' + sample_label + '.pkl'))
vessels_poly = joblib.load(os.path.join(dist_out_dir, 'vessels_poly_' + sample_label + '.pkl'))
for p in tls_poly.geoms:
        ax.plot(*p.exterior.xy, c='black', linewidth=4)
        
plt.xlim([3700, 4400])
plt.ylim([4200, 4900])
# plt.savefig(os.path.join(plot_out_dir, "distance_day3_tls_zoom.png"), dpi=300, bbox_inches='tight', transparent=True)

## Plot CD4 subset distribution IMAPs

In [ ]:
# x_axis = avg_distance_to_bronchi_zone, y_axis = distance_to_tls
sample_label = 'HDM_day3'
adata_plot = adata[adata.obs['sample_label'] == sample_label, :]
y_min = np.min(adata_plot.obs['distance_to_tls'])

gates = {
    'in-TLS': {'y_min':y_min, 'y_max': 1}, # in TLS
    'near-TLS':  {'y_min': 1, 'y_max': 100}, # close to TLS (any bronchi dist)
    'near-Br': {'x_max': 150, 'y_min': 150},      # close to bronchi, far from TLS
    'far':   {'x_min': 200, 'y_min': 150},      # far from both
}

# check cell distributions on IMAP
p = plot_celltype_density_2d(
    adata_plot,
    sample_label=[sample_label],
    # celltype='Th0',
    # celltype_col='label_fine',
    celltype='CD4 act',
    celltype_col='label_medium',  
    x_axis='avg_distance_to_bronchi_zone',
    y_axis='distance_to_tls',
    gates=gates,
    x_clip=450,
    y_clip=450,
    point_size=24,
    max_cells=5000
)
# plt.savefig(os.path.join(plot_out_dir, f"2D_density_CD4act_gates_{sample_label}.pdf"), dpi=300, bbox_inches='tight', transparent=True)
p.show()

## Plot gene expression over distance to TLS and bronchi 
Plot convolved gene expression over distance using scVelo plotting.

In [ ]:
# matplotlib settings
plt.rcdefaults()
mpl.rcParams['pdf.fonttype'] = 42

In [ ]:
adata_d3 = adata[adata.obs['sample_label']=='HDM_day3', :]
adata_d30 = adata[adata.obs['sample_label']=='HDM_day30', :]

In [ ]:
# first make a quick colorbar
import matplotlib.cm as cm
# Define the colormap
cmap = cm.viridis  # or e.g., cm.viridis_r, cm.Reds, etc.

# Create a dummy scalar mappable for the colorbar
norm = mcolors.Normalize(vmin=0, vmax=1)
sm = cm.ScalarMappable(norm=norm, cmap=cmap)

# Create a new figure and axis
fig, ax = plt.subplots(figsize=(1, 4))  # adjust size as needed

# Add the colorbar to the axis
cb = fig.colorbar(sm, cax=ax)

# Remove ticks and labels
cb.ax.set_yticks([])
cb.ax.set_yticklabels([])

# Remove outline
cb.outline.set_visible(False)

# Save the figure
fig.savefig(os.path.join(plot_out_dir, "colormap_viridis_clean.png"), dpi=300, bbox_inches='tight', transparent=True)
plt.close(fig)

In [ ]:
# Filter genes by those robustly expressed in CITE dataset and in Xenium datast (5% threshold in at least one subset):
# CITE genes expressed in 5% of at least one subset
gene_df = pd.read_csv('/home/workspace/spatial_mouse_lung_outputs/distance_analysis/ouputs_mouselung_xenseg_HDM/R/cite_expressed_genes_by_annotation.csv')
genes_cite_filt = gene_df['x'].tolist()

# Xenium genes expressed in 5% of at least one subset
genes_xen_filt = []
# adata_cd4 = adata.copy()
adata_cd4 = adata[adata.obs['label_coarse']=='CD4 T cell', :].copy()
for subset in adata_cd4.obs['label_fine'].unique().tolist():
    adata_cd4_subset = adata_cd4[adata_cd4.obs['label_fine'] == subset, :]
    adata_cd4_subset = du.filter_adata_expressed_in_n_cells(adata_cd4_subset, fraction=0.05)
    genes_xen_filt.extend(adata_cd4_subset.var.index.tolist())
genes_xen_filt = list(set(genes_xen_filt))

# take the intersection and use this list for heatmaps 
genes_cite_xen_filt = list(set(genes_cite_filt) & set(genes_xen_filt))

In [ ]:
genes_highlight_tls = ['Il21', 'Ccr7', 'Slamf6', 'Tcf7', 'Id3', 'Tbx21', 'Cxcr6', 'Il1rl1', 'Gata3', 'Il13', 'Calca']
genes_highlight_bronchi = ['Il1rl1', 'Il2ra', 'Foxp3', 'Gata3', 'Klrg1', 'Nmur1', 'Prdm1', 'Cxcr6', 'Id3', 'Slamf6', 'Tcf7', 'Tbx21', 'Ccr7']

In [ ]:
for sample_label in adata.obs['sample_label'].cat.categories:
    print(sample_label)
    adata_plot = adata[adata.obs['sample_label']==sample_label, :]
    s = scvelo_heatmap(
        adata_plot[:,genes_cite_xen_filt],
        sample_label=[sample_label],
        sortby = 'distance_to_tls',
        key_name = 'label_medium',
        key_value = 'CD4 act',
        highlight = genes_highlight_tls,
        x_clip = 450, 
        expression_threshold= None, # already using a filtered gene list 
        figsize=(5,6)
    )
    s
    plt.savefig(os.path.join(plot_out_dir, f"{sample_label}_scVelo_tls-structure_CD4.pdf"), dpi=300, bbox_inches='tight', transparent=True)

In [ ]:
for sample_label in adata.obs['sample_label'].cat.categories:
    print(sample_label)
    adata_plot = adata[adata.obs['sample_label']==sample_label, :]
    s = scvelo_heatmap(
        adata_plot[~adata_plot.obs['zone_consol'].isin(['TLS']),genes_cite_xen_filt],
        sample_label=[sample_label],
        sortby = 'avg_distance_to_bronchi_zone',
        key_name = 'label_medium',
        key_value = 'CD4 act',
        highlight = genes_highlight_bronchi,
        x_clip = 450, 
        expression_threshold= None, # already using a filtered gene list 
        figsize=(5,6)
    )
    s
    plt.savefig(os.path.join(plot_out_dir, f"{sample_label}_scVelo_bronchi-zone_CD4.pdf"), dpi=300, bbox_inches='tight', transparent=True)

## Ligand–receptor spatial co-localization (Jensen-Shannon distance)

In [ ]:
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance


def _gene_expr(ad, gene, layer=None):
    """1D expression vector for a single gene (handles sparse / layer)."""
    X = ad[:, gene].layers[layer] if layer is not None else ad[:, gene].X
    return (X.toarray() if hasattr(X, 'toarray') else np.asarray(X)).ravel()


def _spatial_field(x, y, w, xedges, yedges, sigma):
    """2D summed-expression grid per bin, optionally Gaussian-smoothed.
    This is the binned data that (after normalization) feeds the JS distance."""
    w = np.clip(np.nan_to_num(w), 0, None)
    H, _, _ = np.histogram2d(x, y, bins=[xedges, yedges], weights=w)
    if sigma:
        H = gaussian_filter(H, sigma=sigma)
    return H


def _normalize(H):
    """Flatten a 2D field to a probability vector (sum 1); None if empty."""
    if H is None:
        return None
    s = H.sum()
    return None if s <= 0 else (H / s).ravel()


def _wasserstein2d_sliced(Pgrid, Qgrid, xcenters, ycenters, n_proj=48, seed=0):
    """Sliced 2D Wasserstein distance between two non-negative grids (same shape).
    Projects bin centers onto random directions and averages the 1D Wasserstein
    distance. Returned in the units of the bin coordinates (um). Unlike JS, this
    accounts for how FAR apart the mass is. None if either grid has no mass."""
    if Pgrid is None or Qgrid is None:
        return None
    p, q = Pgrid.ravel().astype(float), Qgrid.ravel().astype(float)
    if p.sum() <= 0 or q.sum() <= 0:
        return None
    XX, YY = np.meshgrid(xcenters, ycenters, indexing='ij')
    coords = np.column_stack([XX.ravel(), YY.ravel()])
    keep = (p > 0) | (q > 0)                 # only bins with mass (speed)
    coords, p, q = coords[keep], p[keep], q[keep]
    rng = np.random.default_rng(seed)
    thetas = rng.normal(size=(n_proj, 2))
    thetas /= np.linalg.norm(thetas, axis=1, keepdims=True)
    dists = [wasserstein_distance(coords @ t, coords @ t, p, q) for t in thetas]
    return float(np.mean(dists))


def plot_binned_fields(fields, receptor, ligands, celltypes,
                       ncols=4, panel_size=2.2, cmap='viridis'):
    """Diagnostic: spatial maps of the binned summed expression that go into the
    JS calculation -- one panel per ligand (all cells) and one per receptor x
    cell type. Each panel is scaled to its own 99th percentile."""
    xedges, yedges = fields['xedges'], fields['yedges']
    extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
    panels = [(f'ligand: {lig}', fields['ligands'][lig]) for lig in ligands]
    panels += [(f'{receptor} in {ct}', fields['receptors'][ct]) for ct in celltypes]

    n = len(panels)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, dpi=150, squeeze=False,
                             figsize=(panel_size * ncols, panel_size * nrows))
    for ax in axes.ravel():
        ax.axis('off')
    for k, (title, grid) in enumerate(panels):
        ax = axes.ravel()[k]
        ax.axis('on')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(title, fontsize=7)
        if grid is None or not np.any(grid > 0):
            ax.text(0.5, 0.5, 'no signal', ha='center', va='center',
                    fontsize=8, transform=ax.transAxes)
            continue
        vmax = np.percentile(grid[grid > 0], 99)
        ax.imshow(grid.T, origin='lower', extent=extent, aspect='equal',
                  cmap=cmap, vmax=vmax)
    fig.suptitle(f"Binned summed expression (JS inputs) — {fields['sample']}",
                 fontsize=10)
    fig.tight_layout()
    return fig


def plot_ligand_receptor_js_proximity(
    adata, receptor, ligands, celltype_col, celltypes,
    sample='HDM_day3', sample_col='sample_label',
    bin_size_um=50, smooth_sigma=1.0,
    x_coord='x_centroid', y_coord='y_centroid',
    layer=None, cmap='viridis_r', annotate=True, figsize=None,
    show_binned=False, vmin=None, vmax=None, autoscale=False,
    metric='js', n_proj=48, wass_seed=0,
):
    """
    Spatial co-localization of ligand-producing cells and receptor+ cells, per
    cell type. One receptor (-> top bar) and its ligand(s) (-> rows) vs. an
    ordered list of cell types (-> columns).

    Each heatmap cell compares (a) the spatial distribution of the ligand's
    expression over ALL cells and (b) the spatial distribution of `receptor`
    expression within that column's cell type. Both are sum-per-bin histograms
    on a shared `bin_size_um` grid over (x_coord, y_coord), Gaussian-smoothed.

    metric : 'js' (default) or 'wasserstein'
        'js'          -> Jensen-Shannon distance in [0, 1]; bin-independent
                         (ignores how far apart non-overlapping bins are; the
                         smoothing is what gives it any neighbor-awareness).
        'wasserstein' -> sliced 2D Wasserstein distance in um; accounts for how
                         far mass must move, so "near misses" score lower than
                         "far misses". Unbounded; vmax auto-set from the data.
    Both follow 0 = co-localized; with cmap='viridis_r', bright = co-localized.

    Set show_binned=True to also plot the binned summed-expression maps. Returns
    (fig, M, fields): M = [ligand x celltype] distance matrix; fields = dict of
    the 2D binned grids (+ grid edges) for replotting.
    """
    if metric not in ('js', 'wasserstein'):
        raise ValueError("metric must be 'js' or 'wasserstein'")
    missing = [g for g in [receptor, *ligands] if g not in adata.var_names]
    if missing:
        raise ValueError(f'genes not in adata.var_names: {missing}')

    ad = adata[adata.obs[sample_col] == sample]
    x_all = ad.obs[x_coord].to_numpy(dtype=float)
    y_all = ad.obs[y_coord].to_numpy(dtype=float)

    # shared grid edges from all cells in the sample
    xedges = np.arange(x_all.min(), x_all.max() + bin_size_um, bin_size_um)
    yedges = np.arange(y_all.min(), y_all.max() + bin_size_um, bin_size_um)
    xcenters = (xedges[:-1] + xedges[1:]) / 2
    ycenters = (yedges[:-1] + yedges[1:]) / 2

    # ligand fields (all cells), one per row
    lig_fields = {lig: _spatial_field(x_all, y_all, _gene_expr(ad, lig, layer),
                                      xedges, yedges, smooth_sigma)
                  for lig in ligands}

    # receptor field per cell type (column) + mean receptor expression
    rec_all = _gene_expr(ad, receptor, layer)
    rec_fields, mean_recep = {}, {}
    for ct in celltypes:
        mask = (ad.obs[celltype_col] == ct).to_numpy()
        if mask.sum() == 0:
            rec_fields[ct], mean_recep[ct] = None, np.nan
            continue
        rec_fields[ct] = _spatial_field(x_all[mask], y_all[mask], rec_all[mask],
                                        xedges, yedges, smooth_sigma)
        mean_recep[ct] = rec_all[mask].mean()

    fields = {'ligands': lig_fields, 'receptors': rec_fields,
              'xedges': xedges, 'yedges': yedges,
              'bin_size_um': bin_size_um, 'sample': sample}

    # distance matrix [ligand x celltype]
    P = {lig: _normalize(lig_fields[lig]) for lig in ligands}
    Q = {ct: _normalize(rec_fields[ct]) for ct in celltypes}
    M = np.full((len(ligands), len(celltypes)), np.nan)
    for i, lig in enumerate(ligands):
        for j, ct in enumerate(celltypes):
            if metric == 'js':
                if P[lig] is not None and Q[ct] is not None:
                    M[i, j] = jensenshannon(P[lig], Q[ct], base=2)
            else:  # wasserstein
                d = _wasserstein2d_sliced(lig_fields[lig], rec_fields[ct],
                                          xcenters, ycenters,
                                          n_proj=n_proj, seed=wass_seed)
                if d is not None:
                    M[i, j] = d

    # color range / labels depend on metric
    finite = M[np.isfinite(M)]
    if autoscale:
        # stretch the colormap over the actual min..max of the computed values
        vmin = float(finite.min()) if finite.size else 0.0
        vmax = float(finite.max()) if finite.size else 1.0
        if vmax <= vmin:                 # guard against a flat/degenerate range
            vmax = vmin + 1e-9
    else:
        if vmin is None:
            vmin = 0.0
        if vmax is None:
            vmax = 1.0 if metric == 'js' else (float(finite.max()) if finite.size else 1.0)

    if metric == 'js':
        cbar_label = 'J-S distance\n(0 = co-localized)'
        fmt = '{:.2f}'
    else:
        cbar_label = 'Wasserstein dist (um)\n(0 = co-localized)'
        fmt = '{:.0f}'

    # ---- main heatmap + receptor bar ----
    n_row, n_col = len(ligands), len(celltypes)
    if figsize is None:
        figsize = (max(4, 0.7 * n_col + 2), 0.7 * n_row + 2.5)

    fig = plt.figure(figsize=figsize, dpi=150)
    gs = fig.add_gridspec(2, 2, height_ratios=[1.2, max(1, n_row)],
                          width_ratios=[40, 2], hspace=0.05, wspace=0.05)
    ax_bar = fig.add_subplot(gs[0, 0])
    ax_hm = fig.add_subplot(gs[1, 0], sharex=ax_bar)
    cax = fig.add_subplot(gs[1, 1])

    ax_bar.bar(np.arange(n_col), [mean_recep[ct] for ct in celltypes],
               color='0.4', width=0.8)
    ax_bar.set_ylabel(f'mean\n{receptor}', fontsize=8)
    ax_bar.tick_params(labelbottom=False)
    ax_bar.spines[['top', 'right']].set_visible(False)

    cmap_obj = (cmap if isinstance(cmap, mcolors.Colormap)
                else plt.get_cmap(cmap)).copy()
    cmap_obj.set_bad('lightgray')
    im = ax_hm.imshow(np.ma.masked_invalid(M), aspect='auto', cmap=cmap_obj,
                      vmin=vmin, vmax=vmax)
    ax_hm.set_xticks(np.arange(n_col))
    ax_hm.set_xticklabels(celltypes, rotation=90)
    ax_hm.set_yticks(np.arange(n_row))
    ax_hm.set_yticklabels(ligands)
    ax_hm.set_xlim(-0.5, n_col - 0.5)

    if annotate:
        mid = (vmin + vmax) / 2
        for i in range(n_row):
            for j in range(n_col):
                if not np.isnan(M[i, j]):
                    ax_hm.text(j, i, fmt.format(M[i, j]), ha='center', va='center',
                               fontsize=6,
                               color='black')# if M[i, j] > mid else 'black')

    cb = fig.colorbar(im, cax=cax)
    cb.set_label(cbar_label, fontsize=8)
    ax_bar.set_title(f'{receptor} — {sample}', fontsize=11)

    if show_binned:
        plot_binned_fields(fields, receptor, ligands, celltypes)

    return fig, M, fields


In [ ]:
# unpack cell type expression of these DE ligands in each region
sample_label = 'HDM_day3'
adata_select = adata[adata.obs['sample_label']==sample_label,:]
sc.pl.matrixplot(adata_select, var_names = ['Cxcl16','Ccl19', 'Ccl21a', 'Cxcl13'], groupby='label_fine', standard_scale = 'var', swap_axes=False)

In [ ]:
celltypes = ['Th0', 'Th1', 'Th2', 'Th17', 'Treg']
fig, M, fields = plot_ligand_receptor_js_proximity(
    adata,
    receptor='Ccr5',
    ligands=['Ccl3'],
    celltype_col='label_fine',
    celltypes=celltypes,
    sample='HDM_day3',
    bin_size_um=50,
    # vmin=0.5,
    cmap = 'Blues_r',
    autoscale=True,
    # show_binned=True,      
)
plt.show()

# or replot the inputs later without recomputing:
# plot_binned_fields(fields, 'Ccr7', ['Ccl19', 'Ccl21a'], celltypes)


In [ ]:
fig, M, fields = plot_ligand_receptor_js_proximity(
    adata,
    receptor='Cxcr5',
    ligands=['Cxcl13'],
    celltype_col='label_fine',
    celltypes=celltypes,
    sample='HDM_day3',
    autoscale=True,
    bin_size_um=50,
    cmap='Blues_r',     
)
plt.show()

In [ ]:
fig, M, fields = plot_ligand_receptor_js_proximity(
    adata,
    receptor='Cxcr6',
    ligands=['Cxcl16'],
    celltype_col='label_fine',
    celltypes=celltypes,
    sample='HDM_day3',
    # vmin = 0.5,
    autoscale=True,
    cmap = 'Blues_r',
    bin_size_um=50,
    # show_binned=True,   
)
plt.show()

In [ ]:
fig, M, fields = plot_ligand_receptor_js_proximity(
    adata,
    receptor='Ccr6',
    ligands=['Ccl20'],
    celltype_col='label_fine',
    celltypes=celltypes,
    sample='HDM_day3',
    # vmin = 0.5,
    autoscale=True,
    cmap = 'Blues_r',
    bin_size_um=50,
    # show_binned=True,   
)
plt.show()

### Rank ligand–receptor pairs by ΔJS between two cell types

In [ ]:
import re


def _parse_lr(name2):
    """Parse CellChatDB `interaction_name_2` (e.g. 'Tgfb1 - (Tgfbr1+Tgfbr2)')
    into (ligand_genes, receptor_genes) gene-symbol lists. None if unparseable."""
    parts = re.split(r'\s+-\s+', str(name2), maxsplit=1)
    if len(parts) != 2:
        return None
    def genes(s):
        return [g.strip() for g in s.strip().strip('()').split('+') if g.strip()]
    return genes(parts[0]), genes(parts[1])


def rank_lr_js_delta(
    adata, df_lr, celltypes, celltype_col='label_fine',
    sample='HDM_day3', sample_col='sample_label',
    bin_size_um=50, smooth_sigma=1.0,
    x_coord='x_centroid', y_coord='y_centroid',
    min_frac_pos=0.05, layer=None,
):
    """
    For every single-gene-ligand x single-receptor pair, compute the JS
    co-localization distance between the ligand's spatial field (all cells) and
    the receptor's spatial field within each of two cell types, then the signed
    delta.

    Pairs are parsed from `interaction_name_2`. Ligands are kept only if they are
    single genes. Multi-subunit RECEPTOR complexes are EXPANDED: each subunit
    becomes its own (ligand, receptor) pair, tested independently (no co-expression
    requirement). Gene matching to adata.var_names is case-insensitive.

    celltypes : [A, B]
        delta = JS_A - JS_B. Since low JS = co-localized, delta > 0 => the pair is
        MORE co-localized with B than A. (e.g. ['Th0','Th2']: delta>0 = more
        co-localized with Th2.)
    min_frac_pos : float
        Keep a pair only if the receptor is detected (expr > 0) in >= this
        fraction of cells in AT LEAST ONE of the two cell types.

    Returns a DataFrame sorted by delta (desc), with JS_A/JS_B, delta, receptor
    %positive and mean expression per cell type, ligand mean expression, and
    per-cell-type cell counts.
    """
    if len(celltypes) != 2:
        raise ValueError('celltypes must be exactly two: [A, B]')
    ca, cb = celltypes

    # case-insensitive resolver: lowercased symbol -> canonical var name
    var_lower = {v.lower(): v for v in adata.var_names}
    def _resolve(g):
        return var_lower.get(g.lower())

    # build pairs: single-gene ligand x each receptor subunit (dedup, canonical names)
    pairs, seen = [], set()
    for name2 in df_lr['interaction_name_2'].dropna():
        parsed = _parse_lr(name2)
        if parsed is None:
            continue
        ligs, recs = parsed
        if len(ligs) != 1:                       # single-gene ligands only
            continue
        lig = _resolve(ligs[0])
        if lig is None:
            continue
        for rec_raw in recs:                     # expand: one pair per receptor subunit
            rec = _resolve(rec_raw)
            if rec is not None and (lig, rec) not in seen:
                seen.add((lig, rec)); pairs.append((lig, rec))
    if not pairs:
        raise ValueError('no ligand-receptor pairs with both genes in adata.var_names')

    ad = adata[adata.obs[sample_col] == sample]
    x_all = ad.obs[x_coord].to_numpy(float)
    y_all = ad.obs[y_coord].to_numpy(float)
    xedges = np.arange(x_all.min(), x_all.max() + bin_size_um, bin_size_um)
    yedges = np.arange(y_all.min(), y_all.max() + bin_size_um, bin_size_um)

    masks = {c: (ad.obs[celltype_col] == c).to_numpy() for c in (ca, cb)}
    n_cells = {c: int(masks[c].sum()) for c in (ca, cb)}

    # ligand fields (all cells), cached per unique ligand
    lig_norm, mean_lig = {}, {}
    for lig in {p[0] for p in pairs}:
        e = _gene_expr(ad, lig, layer)
        lig_norm[lig] = _normalize(_spatial_field(x_all, y_all, e, xedges, yedges, smooth_sigma))
        mean_lig[lig] = float(e.mean())

    # receptor fields per (receptor, cell type), cached
    rec_norm, rec_stats = {}, {}
    for rec in {p[1] for p in pairs}:
        e_all = _gene_expr(ad, rec, layer)
        for c in (ca, cb):
            m = masks[c]
            if m.sum() == 0:
                rec_norm[(rec, c)], rec_stats[(rec, c)] = None, (0.0, np.nan)
                continue
            ec = e_all[m]
            rec_stats[(rec, c)] = (float((ec > 0).mean()), float(ec.mean()))
            rec_norm[(rec, c)] = _normalize(
                _spatial_field(x_all[m], y_all[m], ec, xedges, yedges, smooth_sigma))

    rows = []
    for lig, rec in pairs:
        fa, fb = rec_stats[(rec, ca)][0], rec_stats[(rec, cb)][0]
        if max(fa, fb) < min_frac_pos:          # receptor floor (receptor only)
            continue
        L, Pa, Pb = lig_norm[lig], rec_norm[(rec, ca)], rec_norm[(rec, cb)]
        if L is None or Pa is None or Pb is None:
            continue
        js_a = jensenshannon(L, Pa, base=2)
        js_b = jensenshannon(L, Pb, base=2)
        rows.append({
            'ligand': lig, 'receptor': rec,
            f'JS_{ca}': js_a, f'JS_{cb}': js_b, 'delta': js_a - js_b,
            f'frac_pos_{ca}': fa, f'frac_pos_{cb}': fb,
            f'mean_recep_{ca}': rec_stats[(rec, ca)][1],
            f'mean_recep_{cb}': rec_stats[(rec, cb)][1],
            'mean_lig': mean_lig[lig],
            f'n_{ca}': n_cells[ca], f'n_{cb}': n_cells[cb],
        })

    res = pd.DataFrame(rows)
    if len(res):
        res = res.sort_values('delta', ascending=False).reset_index(drop=True)
    return res



def plot_js_delta_jitter(res, top_n=5, delta_col='delta',
                         label_cols=('ligand', 'receptor'),
                         figsize=(3.2, 5), jitter=0.08, point_size=14,
                         label_fontsize=7, random_state=0, title=None):
    """Jitter plot of ΔJS per ligand-receptor pair (one point each), y = delta.
    Labels the top_n most positive and top_n most negative as 'ligand→receptor'."""
    d = res.dropna(subset=[delta_col]).copy()
    y = d[delta_col].to_numpy()
    rng = np.random.default_rng(random_state)
    x = rng.uniform(-jitter, jitter, len(y))

    ymin, ymax = y.min(), y.max()
    pad = 0.05 * (ymax - ymin if ymax > ymin else 1.0)
    ylo, yhi = ymin - pad, ymax + pad
    min_gap = (yhi - ylo) / 30.0

    def _declutter(t, gap, lo, hi):
        t = np.asarray(t, float)
        idx = np.argsort(t)
        ys = t[idx].copy()
        for i in range(1, len(ys)):
            if ys[i] - ys[i - 1] < gap:
                ys[i] = ys[i - 1] + gap
        ys += t.mean() - ys.mean()
        if ys[-1] > hi:
            ys -= ys[-1] - hi
        if ys[0] < lo:
            ys -= ys[0] - lo
        out = np.empty_like(ys)
        out[idx] = ys
        return out

    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    ax.scatter(x, y, s=point_size, color='0.6', alpha=0.7, edgecolors='none', zorder=1)

    d_sorted = d.sort_values(delta_col, ascending=False)
    top = d_sorted.head(top_n)
    bot = d_sorted.tail(top_n)
    bot = bot[~bot.index.isin(top.index)]          # avoid overlap on tiny tables
    lc0, lc1 = label_cols
    lx = jitter + 0.05
    for grp in (top, bot):
        ax.scatter(np.zeros(len(grp)), grp[delta_col], s=point_size + 10,
                   color='black', zorder=2)
        ly = _declutter(grp[delta_col].to_numpy(), min_gap, ylo, yhi)
        for (_, row), yl in zip(grp.iterrows(), ly):
            ax.annotate(f"{row[lc0]}\u2192{row[lc1]}",
                        xy=(0, row[delta_col]), xytext=(lx, yl),
                        fontsize=label_fontsize, va='center', ha='left',
                        arrowprops=dict(arrowstyle='-', lw=0.4, color='0.6'))

    ax.axhline(0, color='black', lw=0.8, ls='--', zorder=0)
    ax.set_xticks([])
    ax.set_xlim(-0.4, 1.6)
    ax.set_ylim(ylo, yhi)
    ax.set_ylabel('Δ JS distance (' + '−'.join(c[3:] for c in res.columns if c.startswith('JS_')) + ')')
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
    if title:
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    return fig, ax


In [ ]:
df_lr = pd.read_csv(os.path.join(inputs_dir, 'interaction_input_CellChatDB.csv'))

res = rank_lr_js_delta(
    adata, df_lr,
    celltypes=['Th0', 'Th2'],      # delta = JS_Th0 - JS_Th2  (>0 = more co-localized with Th2)
    celltype_col='label_fine',
    sample='HDM_day3',
    bin_size_um=50, smooth_sigma=1.0,
    min_frac_pos=0.10,
)
print(res.shape)
display(res.head(15))   # top: more co-localized with Th2
display(res.tail(15))   # bottom: more co-localized with Th0
res.to_csv('Th0_Th2_deltaJS.csv')

In [ ]:
fig, ax = plot_js_delta_jitter(res, top_n=5)
plt.show()


In [ ]:
df_lr = pd.read_csv(os.path.join(inputs_dir, 'interaction_input_CellChatDB.csv'))

res_th17 = rank_lr_js_delta(
    adata, df_lr,
    celltypes=['Th17', 'Th2'],      # delta = JS_Th0 - JS_Th2  (>0 = more co-localized with Th2)
    celltype_col='label_fine',
    sample='HDM_day3',
    bin_size_um=50, smooth_sigma=1.0,
    min_frac_pos=0.10,
)

fig, ax = plot_js_delta_jitter(res_th17, top_n=5)
plt.show()


## Plot gene density in 2D plots

In [ ]:
sample_label = 'HDM_day3'
print(sample_label)
adata_plot = adata[adata.obs['sample_label']==sample_label, :]
# for gene_name in ['Ccr7', 'Tgfbr1', 'Tgfbr2','Icos', 'Pdcd1' ]:
for gene_name in ['Ccr7', 'Cxcr5', 'Cxcr6']:
    p = distance_xy_kde(adata_plot,
                    sample_label=[sample_label],
                    x_axis = 'avg_distance_to_bronchi_zone',
                        y_axis = 'distance_to_tls',
                        gene_name = gene_name,
                    celltype_col = 'label_medium',
                    celltype = 'CD4 act',
                   x_clip = 450,
                   y_clip= 450,
                        gates = gates
                       )


In [ ]:
import session_info
print('active conda environment: ', os.path.basename(sys.prefix))
session_info.show()